# Proyecto 2 - VectorIA
## Módulo 1: Detección y Clasificación del Mosquito Vector (CNN Propia vs MobileNetV2)

**Objetivo:** Clasificar fotografías de mosquitos en 3 especies clave para la salud pública:
1. `Aedes aegypti` (Vector principal del dengue)
2. `Aedes albopictus` (Vector secundario)
3. `Culex` (No transmite dengue)

**Rúbrica:**
- Implementar una **CNN propia desde cero**.
- Implementar **MobileNetV2 con Transfer Learning** (Fase 1: Feature Extractor, Fase 2: Fine-Tuning).
- Reportar el F1-Score Macro de ambas arquitecturas (Meta $\ge 85\%$).

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from src.train.cnn import EntrenarCNN, DiagnosticoEvaluacion
from src.train.cnn_mobilenetv2 import EntrenarMobileNetV2

## PARTE A: CNN Propia (Entrenada desde cero)
Arquitectura con bloques convolucionales dobles, BatchNormalization, Dropout y GlobalAveragePooling2D.

In [ ]:
data_path = '../data/processed'
modelo_cnn = EntrenarCNN(data_dir=data_path, img_size=(224, 224), batch_size=32)
modelo_cnn.cargar_datos()

In [ ]:
modelo_cnn.construir_modelo()
pesos_cnn = modelo_cnn.calcular_pesos_clases()

In [ ]:
historia_cnn = modelo_cnn.compilar_y_entrenar(epochs=30, class_weights=pesos_cnn)
modelo_cnn.guardar_modelo()

In [ ]:
eval_cnn = DiagnosticoEvaluacion(
    modelo=modelo_cnn.modelo,
    historia=historia_cnn,
    test_ds=modelo_cnn.test_ds,
    class_names=modelo_cnn.class_names
)
eval_cnn.graficar_metricas()
eval_cnn.evaluar_test()

## PARTE B: MobileNetV2 (Transfer Learning & Fine-Tuning)
Modelo pre-entrenado en ImageNet adaptado con cabeza densa personalizada y fine-tuning en las últimas 60 capas.

In [ ]:
modelo_mobilenet = EntrenarMobileNetV2(data_dir=data_path, img_size=(224, 224), batch_size=32)
modelo_mobilenet.cargar_datos()
modelo_mobilenet.construir_modelo()
pesos_mobile = modelo_mobilenet.calcular_pesos_clases()

In [ ]:
# Fase 1: Extracción de características (cuerpo congelado)
hist_fase1 = modelo_mobilenet.entrenar_fase1(epochs=15, class_weights=pesos_mobile)

In [ ]:
# Fase 2: Fine-Tuning (descongelando últimas 60 capas con learning rate 1e-5)
hist_fase2 = modelo_mobilenet.entrenar_fine_tuning(epochs_adicionales=20, capas_a_descongelar=60, class_weights=pesos_mobile)
modelo_mobilenet.guardar_modelo()

In [ ]:
eval_mobile = DiagnosticoEvaluacion(
    modelo=modelo_mobilenet.modelo,
    historia=hist_fase2,
    test_ds=modelo_mobilenet.test_ds,
    class_names=modelo_mobilenet.class_names
)
eval_mobile.graficar_metricas()
eval_mobile.evaluar_test()